In [1]:
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import editdistance
from collections import defaultdict
import tqdm
rats = pd.read_csv(r"C:\Users\music\Desktop\CompLab\DesignFiles\SeqFeat\SynADs_seqfeat.csv")
synthseq = rats[rats['Ctrl'] == 0]
synthseq = synthseq[synthseq['NegCtrl'] == 0]
choptf = pd.read_csv(r'C:\Users\music\Desktop\CompLab\DesignFiles\SeqFeat\ChopTF_seqfeat.csv')
train = pd.read_csv(r'C:\Users\music\Desktop\CompLab\SynAD\data\Training\TrainingSeqs_seqfeats.csv', index_col=0)
train = train[train['AD Origin'] == 'More_Plant_data.PADI_library.xlsx']
prot = pd.read_csv(r"C:\Users\music\Desktop\CompLab\Global_Data\TiledProteome.csv")

In [2]:
def min_editdist_to_df(df1, df2, seq_col1='AAseq', seq_col2='AAseq',
                       out_col='min_edit_dist', match_col='match_AAseq', use_tqdm=True):
    """
    For each sequence in df1[seq_col1], compute the minimum edit distance to any
    sequence in df2[seq_col2]. Returns a copy of df1 with column `out_col`
    (int or np.nan). Also stores the matching df2 sequence in `match_col`.
    Uses length-based buckets to short-circuit comparisons for speed.
    """
    if df2 is None or len(df2) == 0:
        out = df1.copy()
        out[out_col] = np.nan
        out[match_col] = None
        return out

    # Build length -> list of unique sequences mapping for df2
    seqs2 = df2[seq_col2].astype(str).dropna().unique()
    len_buckets = defaultdict(list)
    for s in seqs2:
        len_buckets[len(s)].append(s)
    sorted_lengths = sorted(len_buckets.keys())

    idx_iter = df1.index
    if use_tqdm:
        try:
            from tqdm.auto import tqdm
            idx_iter = tqdm(df1.index, desc='min_editdist')
        except Exception:
            idx_iter = df1.index

    out = df1.copy()
    mins = []
    matches = []

    for i in idx_iter:
        s1 = str(df1.at[i, seq_col1]) if pd.notna(df1.at[i, seq_col1]) else ''
        len1 = len(s1)
        best = np.inf
        best_seq = None

        # iterate candidate lengths (small -> large) but skip buckets that cannot improve
        for l in sorted_lengths:
            lb = abs(len1 - l)  # lower bound on edit distance for this length
            if lb >= best:
                # no sequence of this length can improve current best
                continue
            for s2 in len_buckets[l]:
                # quick lower-bound check (recompute lb per bucket not necessary inside loop)
                if lb >= best:
                    break
                d = editdistance.eval(s1, s2)
                if d < best:
                    best = d
                    best_seq = s2
                    if best == 0:
                        break
            if best == 0:
                break

        if np.isinf(best):
            mins.append(np.nan)
            matches.append(None)
        else:
            mins.append(int(best))
            matches.append(best_seq)

    out[out_col] = mins
    out[match_col] = matches

    return out

In [3]:
min_editdist_to_df(df1=synthseq, df2=train, use_tqdm=True)

c:\Users\music\miniforge3\envs\metapred\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
min_editdist: 100%|██████████| 18750/18750 [07:41<00:00, 40.60it/s]


,AAseq,Ctrl,NegCtrl,Net_Charge,Aromatics,WFYL,HydrophobicityKD,HydrophobicityWW,Kappa,Omega,...,P,Q,R,S,T,V,W,Y,min_edit_dist,match_AAseq
0,MLDEEGLLLDEDTEVFSTAEAVSDEEALEDLFNSQNLMSP,0,0,-13.0,2,9,0.464444,-0.47500,0.257697,0.197367,...,1,1,0,4,2,2,0,0,25,DLDGEEDLLSLSDTVVSDDFFDGLEEFAAGDSFSGNSAPA
1,CFALFSDEDELEDDSDIEEISESFWAMELRMNLESLDTIL,0,0,-13.0,4,10,0.477778,-0.39175,0.406074,0.121434,...,0,0,1,5,1,0,1,0,27,FDFSDEFDAESILDEDIEEGIDSIMGTVVESNSNSGIYES
2,RTDLHFDLQEEVDIGLDSFEADILDFAKTDKEFQSLLVKG,0,0,-7.0,4,10,0.461944,-0.37825,0.231213,0.080478,...,0,2,1,2,2,2,0,0,28,KVSLNSDFMENNEVDDGVERFQTLDPIPEAKTVKKSERKR
3,LDRLKELDDFDDHLLSSDVDEDSACYDFTFGLCFSKLFDG,0,0,-9.0,6,13,0.466944,-0.25050,0.302654,0.183703,...,0,0,1,4,1,1,0,1,29,SMDELDFNKDFDLPPSSNQTLGLANGFYLDDLDFSSLDPP
4,SDEQLELSEKQSYSDLDYKAEGLDSFPDAVQLWFPFLLDI,0,0,-8.0,6,13,0.452500,-0.24500,0.148395,0.080848,...,2,3,0,5,0,1,1,2,28,SFICEYAGELLEDKQAESLTGKDEYLFDLGDEDDPFTINA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18745,IDHPSSSSMLSGELLFDGCWLETTDGSEFLSTNSDYDIGI,0,0,-8.0,4,9,0.483056,-0.15725,0.125313,0.212459,...,1,0,0,8,3,0,1,1,22,EDKNNNDVEMDSSMFLSEREEENRFVSEFLSTNSDYDIGI
18746,WSSVTSFVSPDESFFNVSAFSEDEWIEVDSDEDKMLVGDV,0,0,-10.0,6,7,0.476667,-0.29500,0.312324,0.315117,...,1,0,0,8,1,6,2,0,26,VTLEESPDESFFNVSAQQCWEMVMRRVKDTSTSLGLPILP
18747,VKLFLCSDDDLFSAYMNLDNIDHLLDFSSLDPSNYNNMWN,0,0,-6.0,6,13,0.472500,-0.05925,0.306706,0.302964,...,1,0,0,5,0,1,1,2,21,VGEREAMDDLFSAYMNLENIDVLNSSEADDSKNGNENRDD
18748,NQEYKKGLWTVEEDLEWLSNFVEDSFSDDTDEDLYKYDPW,0,0,-10.0,8,12,0.358611,-0.33800,0.354541,0.098684,...,1,1,0,3,2,2,3,3,24,NQEYKKGLWTVEEDNILMDYVLNHGTGQWNRIVRKTGLKR


In [4]:
traindist = min_editdist_to_df(df1=synthseq, df2=train, use_tqdm=True)

min_editdist: 100%|██████████| 18750/18750 [32:16<00:00,  9.68it/s]     


In [5]:
traindist.to_csv(r'C:\Users\music\Desktop\CompLab\SynAD\data\EditDistances\MinSynth_to_Train.csv', index=False)

In [6]:
protdist = min_editdist_to_df(df1=synthseq, df2=prot, use_tqdm=True)
protdist.to_csv(r'C:\Users\music\Desktop\CompLab\SynAD\data\EditDistances\MinSynth_to_Proteome.csv', index=False)

min_editdist: 100%|██████████| 18750/18750 [4:14:11<00:00,  1.23it/s]      


In [7]:
choptfdist = min_editdist_to_df(df1=synthseq, df2=choptf, use_tqdm=True)
choptfdist.to_csv(r'C:\Users\music\Desktop\CompLab\SynAD\data\EditDistances\MinSynth_to_ChopTF.csv', index=False)

min_editdist: 100%|██████████| 18750/18750 [37:49<00:00,  8.26it/s]
